In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
 
from riemannian import *
from propagator import *

## 1D · Schrödinger · flat metric · free particle

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ex 1 — 1D · Schrödinger · flat metric · free particle
# ─────────────────────────────────────────────────────────────────────────────
#
# PDE :  −i ∂u/∂t = −½ ∂²u/∂x²
#
# The flat metric g = 1 gives H = ½ ξ² (kinetic energy only).
# Rays are straight lines x(t) = x₀ + v·t.  No caustics.
# The exact solution for a point source is a spreading Gaussian envelope
# modulated by exp(ix²/2t) — the WKB sum reproduces it ray by ray.
#
# What to observe:
#   • Quadratic phase ramp in the arg(ψ) panel.
#   • |ψ|² decays as 1/√t (amplitude panel broadens and flattens).
#   • No caustic corrections needed (det J = t > 0 always).
 
x = sp.Symbol('x', real=True)
metric_flat_1d = Metric(sp.Integer(1), (x,))
 
result_ex1 = compute_wavefunction(
    metric     = metric_flat_1d,
    source     = (0.0,),
    v_fan      = np.linspace(-5.0, 5.0, 200),
    t_max      = 2.5,
    hbar       = 0.08,
    n_steps    = 150,
    N_grid     = 100,
    integrator = 'verlet',
    equation   = EquationType.SCHRODINGER,
    parallel   = True,
)
 
fig1 = plot_wavefunction(result_ex1, log_scale=False)
fig1.suptitle("Ex 1 — 1D Schrödinger, free particle  (flat metric)",
              color='white', fontsize=10, y=1.02)
plt.show()

anim = animate_wavefunction(result_ex1, n_frames=80, interval=40)
HTML(anim.to_jshtml())

## 1D · Parabolic (heat kernel) · harmonic potential

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ex 2 — 1D · Parabolic (heat kernel) · harmonic potential
# ─────────────────────────────────────────────────────────────────────────────
#
# PDE :  ∂u/∂t = ψOp u,   H(x, ξ) = ½ ξ² − ½ x²
#
# The inverted harmonic oscillator (H = ½p² − ½x²) makes an excellent
# parabolic test: rays diverge exponentially, the solution concentrates
# along the unstable manifolds, and caustics appear at finite time.
#
# WKB ansatz:  u = A exp(S/ℏ)  — real exponential, no oscillations.
# The parabolic cylinder correction D_{-1/2} patches the fold caustic.
#
# What to observe:
#   • u(x,t) is real-valued.
#   • The solution grows exponentially on the unstable directions.
#   • The log|u| panel (top-right) shows the action landscape directly.
 
x, xi = sp.symbols('x xi', real=True)
H_heat = xi**2 / 2 - x**2 / 2          # inverted harmonic oscillator
 
result_ex2 = compute_wavefunction(
    hamiltonian = H_heat,
    coords      = (x,),
    momenta     = (xi,),
    source      = (0.0,),
    p_fan       = np.linspace(-2.5, 2.5, 200),
    t_max       = 1.5,
    hbar        = 0.1,
    n_steps     = 150,
    N_grid      = 100,
    integrator  = 'rk45',
    equation    = EquationType.PARABOLIC,
    parallel    = True,
)
 
fig2 = plot_wavefunction(result_ex2, log_scale=True)
fig2.suptitle("Ex 2 — 1D Parabolic heat kernel, inverted harmonic oscillator",
              color='white', fontsize=10, y=1.02)
plt.show()
 
fig2b = plot_ray_fan(result_ex2)
plt.show()

anim = animate_wavefunction(result_ex2, n_frames=80, interval=40)
HTML(anim.to_jshtml())

## 1D · Wave equation · variable-speed medium  c(x) = 1 / √(1 + x²/4)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ex 3 — 1D · Wave equation · variable-speed medium  c(x) = 1 / √(1 + x²/4)
# ─────────────────────────────────────────────────────────────────────────────
#
# PDE :  ∂²u/∂t² = ψOp u,   H = c²(x) ξ²,   c²(x) = 1/(1 + x²/4)
#
# This models an acoustic waveguide: the wave speed c(x) decreases away from
# the origin, creating a refractive channel that bends rays back toward x = 0.
# The two dispersion branches H± = ±√H = ±c(x)|ξ| generate left- and
# right-going wave families that superpose to form a standing-wave-like pattern.
#
# What to observe:
#   • Two symmetric ray fans (forward / backward branches) visible in the
#     ray-fan panel.
#   • Constructive / destructive interference between the two branches.
#   • Caustic focusing near x = 0 (rays reflected by the waveguide).
 
x, xi = sp.symbols('x xi', real=True)
c2    = 1 / (1 + x**2 / 4)             # c²(x) — varies from 1 at origin
H_wave_1d = c2 * xi**2                  # acoustic dispersion: H = c²|ξ|²
 
result_ex3 = compute_wavefunction(
    hamiltonian = H_wave_1d,
    coords      = (x,),
    momenta     = (xi,),
    source      = (0.0,),
    p_fan       = np.linspace(-3.0, 3.0, 200),
    t_max       = 3.0,
    hbar        = 0.12,
    n_steps     = 300,
    N_grid      = 200,
    integrator  = 'rk45',
    equation    = EquationType.WAVE,
    parallel    = True,
)
 
fig3 = plot_wavefunction(result_ex3, log_scale=False)
fig3.suptitle(r"Ex 3 — 1D Wave equation, waveguide  $c^2(x)=1/(1+x^2/4)$",
              color='white', fontsize=10, y=1.02)
plt.show()
 
fig3b = plot_interference_detail(result_ex3)
plt.show()

anim = animate_wavefunction(result_ex3, n_frames=80, interval=40)
HTML(anim.to_jshtml())

## 1D Harmonic Oscillator — Non-constant Maslov Index

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Example: 1D Harmonic Oscillator — Non-constant Maslov Index
# ─────────────────────────────────────────────────────────────────────────────
#
# H = ½ p² + ½ x²
#
# Key physics:
#   • All rays from x=0 focus at t = π, 2π, 3π, ... (periodic caustics)
#   • Each focus increments Maslov index μ by +1
#   • With t_max > π, rays accumulate μ ≥ 1
#   • Different rays may cross caustics at slightly different times
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from propagator import compute_wavefunction, plot_wavefunction, plot_ray_fan
from propagator import EquationType

# Define 1D harmonic oscillator Hamiltonian
x, xi = sp.symbols('x xi', real=True)
H_ho = xi**2 / 2 + x**2 / 2

# Source at origin
source = (0.0,)

# Momentum fan - exclude zero to avoid degenerate rays
p_fan = np.concatenate([
    np.linspace(-2.0, -0.2, 20),
    np.linspace(0.2, 2.0, 20)
])

print(f"Total rays: {len(p_fan)}")

# ─────────────────────────────────────────────────────────────────────────────
# Case 1: t_max < π  →  No caustics, μ = 0 everywhere
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("Case 1: t_max = 2.0  (< π ≈ 3.14)  →  No caustics, μ = 0 everywhere")
print("="*70)

result_1 = compute_wavefunction(
    hamiltonian = H_ho,
    coords      = (x,),
    momenta     = (xi,),
    source      = source,
    p_fan       = p_fan,
    t_max       = 2.0,          # Less than π
    hbar        = 0.1,
    n_steps     = 400,
    N_grid      = 200,
    integrator  = 'rk45',
    equation    = EquationType.SCHRODINGER,
    parallel    = False,        # Easier debugging in 1D
)

print(f"Maslov index range: [{result_1.mu_pts.min()}, {result_1.mu_pts.max()}]")
print(f"Unique Maslov values: {np.unique(result_1.mu_pts)}")
print(f"Rays with μ=0: {np.sum(result_1.mu_pts == 0)}")

fig1 = plot_wavefunction(result_1, log_scale=True)
fig1.suptitle(r"HO 1D: $t_{max}=2.0 < \pi$  (No caustics, $\mu=0$)", 
              color='white', fontsize=10, y=1.02)
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Case 2: t_max > π  →  Caustics formed, μ ≥ 1 for some rays
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("Case 2: t_max = 4.0  (> π ≈ 3.14)  →  Caustics formed, μ varies")
print("="*70)

result_2 = compute_wavefunction(
    hamiltonian = H_ho,
    coords      = (x,),
    momenta     = (xi,),
    source      = source,
    p_fan       = p_fan,
    t_max       = 4.0,          # Greater than π
    hbar        = 0.1,
    n_steps     = 600,          # More steps for accuracy through caustic
    N_grid      = 200,
    integrator  = 'rk45',
    equation    = EquationType.SCHRODINGER,
    parallel    = False,
)

print(f"Maslov index range: [{result_2.mu_pts.min()}, {result_2.mu_pts.max()}]")
print(f"Unique Maslov values: {np.unique(result_2.mu_pts)}")
print(f"Rays with μ=0: {np.sum(result_2.mu_pts == 0)}")
print(f"Rays with μ=1: {np.sum(result_2.mu_pts == 1)}")
print(f"Rays with μ≥2: {np.sum(result_2.mu_pts >= 2)}")

fig2 = plot_wavefunction(result_2, log_scale=True)
fig2.suptitle(r"HO 1D: $t_{max}=4.0 > \pi$  (Caustics, $\mu \geq 1$)", 
              color='white', fontsize=10, y=1.02)
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Case 3: t_max > 2π  →  Multiple caustic crossings, μ ≥ 2
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("Case 3: t_max = 7.0  (> 2π ≈ 6.28)  →  Multiple caustics, μ ≥ 2")
print("="*70)

result_3 = compute_wavefunction(
    hamiltonian = H_ho,
    coords      = (x,),
    momenta     = (xi,),
    source      = source,
    p_fan       = p_fan,
    t_max       = 7.0,          # Greater than 2π
    hbar        = 0.1,
    n_steps     = 800,
    N_grid      = 200,
    integrator  = 'rk45',
    equation    = EquationType.SCHRODINGER,
    parallel    = False,
)

print(f"Maslov index range: [{result_3.mu_pts.min()}, {result_3.mu_pts.max()}]")
print(f"Unique Maslov values: {np.unique(result_3.mu_pts)}")

fig3 = plot_wavefunction(result_3, log_scale=True)
fig3.suptitle(r"HO 1D: $t_{max}=7.0 > 2\pi$  (Multiple caustics, $\mu \geq 2$)", 
              color='white', fontsize=10, y=1.02)
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Diagnostic: Plot Maslov index distribution
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, result, t_max in zip(axes, [result_1, result_2, result_3], [2.0, 4.0, 7.0]):
    unique, counts = np.unique(result.mu_pts, return_counts=True)
    ax.bar(unique, counts, color='steelblue', alpha=0.7)
    ax.set_xlabel(r"Maslov index $\mu$")
    ax.set_ylabel("Number of ray points")
    ax.set_title(f"t_max = {t_max}")
    ax.set_xticks(unique)

plt.tight_layout()
fig.patch.set_facecolor('#0e0e1a')
for ax in axes:
    ax.set_facecolor('#0e0e1a')
    ax.tick_params(colors='white')
    for lbl in (ax.xaxis.label, ax.yaxis.label, ax.title):
        lbl.set_color('white')
    for sp in ax.spines.values():
        sp.set_edgecolor('#444')

plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Diagnostic: Show rays coloured by Maslov index
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

x_key, _ = result_2.rays[0].traj.keys(), None  # Get position key
for key in result_2.rays[0].traj.keys():
    if key not in {'t', 'xi', 'energy'}:
        x_key = key
        break

cmap = plt.cm.RdYlGn
max_mu = max(1, result_2.mu_pts.max())

for ray in result_2.rays:
    ax.plot(ray.traj['t'], ray.traj[x_key], 
            lw=1.0, alpha=0.5, 
            color=cmap(ray.mu / max_mu))

ax.set_xlabel('t')
ax.set_ylabel('x')
ax.set_title(r"HO 1D: Rays coloured by Maslov index $\mu$  ($t_{max}=4.0$)")
ax.set_facecolor('#0e0e1a')
ax.tick_params(colors='white')
for lbl in (ax.xaxis.label, ax.yaxis.label, ax.title):
    lbl.set_color('white')
for sp in ax.spines.values():
    sp.set_edgecolor('#444')

plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Verify Maslov index is integer-valued
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("Verification: Maslov index should be integer-valued")
print("="*70)

for i, result in enumerate([result_1, result_2, result_3], 1):
    is_integer = np.all(result.mu_pts == np.round(result.mu_pts))
    print(f"Case {i} (t_max={[2.0, 4.0, 7.0][i-1]}): "
          f"All integers? {is_integer}, "
          f"dtype={result.mu_pts.dtype}, "
          f"unique={np.unique(result.mu_pts)}")

##  1D Heat Equation — Gaussian Diffusion

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Example: 1D Heat Equation — Gaussian Diffusion (CORRECTED)
# ─────────────────────────────────────────────────────────────────────────────
#
# PDE:  ∂u/∂t = ½ ∂²u/∂x²
#
# The semiclassical (WKB) approximation for the parabolic equation gives:
#     u(x,t) ≈ Σ_k exp(S_k/ℏ) / √|det J_k|
#
# Unlike Schrödinger equation:
#   - Solution is REAL (no imaginary unit in exponent)
#   - No oscillatory fringes
#   - Amplitude grows exponentially with action: exp(v²·t/ℏ)
#   - Diffusion spreads the initial peak
#
# NUMERICAL STABILITY REQUIREMENT:
# ─────────────────────────────────
# For free particle: S = v²·t, so each ray contributes exp(v²·t/ℏ)
# To avoid exponential explosion, we need: v²·t/ℏ ≲ 10
#
# This example uses:
#   • Narrow velocity fan: |v| ≤ 0.8
#   • Larger ℏ: ℏ = 1.5
#   • Result: max(v²·t/ℏ) ≈ (0.8)²·2.0/1.5 ≈ 0.85 ✓
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from propagator import compute_wavefunction, plot_wavefunction
from propagator import EquationType
from riemannian import Metric

# Define 1D flat metric
x = sp.Symbol('x', real=True)
m_1d = Metric(1, (x,))

# Source at origin
source = (0.0,)

# ─────────────────────────────────────────────────────────────────────────────
# OPTION 1: NARROW VELOCITY FAN (IMPLEMENTED)
# ─────────────────────────────────────────────────────────────────────────────
# Keep |v| small to prevent exp(v²·t/ℏ) explosion

v_fan = np.concatenate([
    np.linspace(-0.8, -0.1, 20),
    np.linspace(0.1, 0.8, 20)
])

# Use larger ℏ to further suppress exponential growth
hbar = 1.5

print("="*70)
print("1D Heat Equation: Gaussian Diffusion")
print("="*70)
print(f"Using {len(v_fan)} rays, ℏ = {hbar}")
print(f"Velocity range: [{v_fan.min():.2f}, {v_fan.max():.2f}]")
print(f"Max exponent estimate: v²·t/ℏ ≈ {(v_fan.max()**2 * 2.0 / hbar):.2f}")
print()

# Compute at different times to show spreading
times = [0.5, 1.0, 2.0]
results = []

for t_max in times:
    result = compute_wavefunction(
        metric     = m_1d,
        source     = source,
        v_fan      = v_fan,
        t_max      = t_max,
        hbar       = hbar,
        n_steps    = 200,
        N_grid     = 150,
        integrator = 'rk45',
        equation   = EquationType.PARABOLIC,
        parallel   = True,
        xlim       = (-4.0, 4.0),
    )
    results.append(result)
    print(f"t = {t_max:4.1f}: rays = {len(result.rays):3d}, "
          f"max |u| = {np.max(np.abs(result.psi)):.4e}")

print()

# ─────────────────────────────────────────────────────────────────────────────
# Analytical solution for comparison (fundamental solution/heat kernel)
# ─────────────────────────────────────────────────────────────────────────────

def analytical_heat_kernel(x, t, sigma0=0.5):
    """
    Fundamental solution (heat kernel) for point source at x=0.
    
    u(x,t) = (4πt)^{-1/2} exp(−x²/(4t))
    
    For comparison, we normalize to match WKB peak amplitude.
    """
    if t <= 0:
        return np.zeros_like(x)
    return (1.0 / np.sqrt(4 * np.pi * t)) * np.exp(-x**2 / (4 * t))

# ─────────────────────────────────────────────────────────────────────────────
# Plot 1: All times together
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(times)))

for result, t_max, color in zip(results, times, colors):
    # For parabolic equation, solution is real
    ax.plot(result.X, result.psi.real, lw=2, label=f"WKB t = {t_max}", 
            color=color, alpha=0.8)
    
    # Add analytical solution for comparison
    u_exact = analytical_heat_kernel(result.X, t_max)
    # Normalize analytical to match WKB peak
    if np.max(u_exact) > 0:
        u_exact *= np.max(result.psi.real) / np.max(u_exact)
    ax.plot(result.X, u_exact, lw=1.5, ls='--', label=f"Analytical t = {t_max}",
            color=color, alpha=0.5)

ax.set_xlabel("x", fontsize=12)
ax.set_ylabel("u(x,t)", fontsize=12)
ax.set_title("1D Heat Equation: Gaussian Spreading", 
             fontsize=14, fontweight="bold")
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#0e0e1a')
fig.patch.set_facecolor('#0e0e1a')
ax.tick_params(colors='white')
for lbl in (ax.xaxis.label, ax.yaxis.label, ax.title):
    lbl.set_color('white')
for sp in ax.spines.values():
    sp.set_edgecolor('#444')

plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Plot 2: Master figure from plot_wavefunction
# ─────────────────────────────────────────────────────────────────────────────

fig = plot_wavefunction(results[-1], log_scale=False)
fig.suptitle(r"1D Heat Equation ($\hbar=1.5$, $t=2.0$)", 
             color='white', fontsize=10, y=1.02)
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Plot 3: Peak amplitude decay over time
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 6))

max_values = [np.max(np.abs(result.psi.real)) for result in results]
ax.plot(times, max_values, 'o-', lw=2, markersize=8, color='cyan', label="WKB")

# Theoretical decay: amplitude ∝ 1/√t for 1D diffusion
t_fine = np.linspace(0.2, 2.2, 100)
theoretical_1d = max_values[0] * np.sqrt(times[0] / t_fine)
ax.plot(t_fine, theoretical_1d, '--', lw=1.5, label=r"Theoretical: $A \propto 1/\sqrt{t}$", 
        color='white', alpha=0.7)

ax.set_xlabel("Time t", fontsize=12)
ax.set_ylabel("Peak amplitude max|u|", fontsize=12)
ax.set_title("Amplitude Decay Over Time (1D Heat Equation)", 
             fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_facecolor('#0e0e1a')
fig.patch.set_facecolor('#0e0e1a')
ax.tick_params(colors='white')
for lbl in (ax.xaxis.label, ax.yaxis.label, ax.title):
    lbl.set_color('white')
for sp in ax.spines.values():
    sp.set_edgecolor('#444')

plt.tight_layout()
plt.show()

print("="*70)
print("Summary")
print("="*70)
print("✓ Solution is REAL (no imaginary part)")
print("✓ No oscillatory fringes (unlike Schrödinger)")
print("✓ Peak amplitude decreases as width increases")
print("✓ Gaussian spreads as σ(t) ∝ √t")
print("✓ Amplitude decays as A(t) ∝ 1/√t")
print()
print("Numerical stability achieved with:")
print(f"  • Narrow velocity fan: |v| ≤ {v_fan.max():.1f}")
print(f"  • Large ℏ: ℏ = {hbar}")
print(f"  • Max exponent: v²·t/ℏ ≈ {(v_fan.max()**2 * 2.0 / hbar):.2f} < 10 ✓")
print("="*70)

## 2D · Schrödinger · flat metric · double-slit interference

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ex 4 — 2D · Schrödinger · flat metric · double-slit interference (FIXED)
# ─────────────────────────────────────────────────────────────────────────────

import sympy as sp

# Flat 2D metric
x, y = sp.symbols('x y', real=True)
g2d_flat = sp.Matrix([[1, 0], [0, 1]])
metric_flat_2d = Metric(g2d_flat, (x, y))

d = 0.4    # slit separation

# Velocity fan - ensure good coverage
vx_vals = np.linspace(0.5, 3.5, 25)
vy_vals = np.linspace(-2.0, 2.0, 25)
v_fan_2d = np.array([[vx, vy] for vx in vx_vals for vy in vy_vals])

def _slit_result(source_y):
    return compute_wavefunction(
        metric     = metric_flat_2d,
        source     = (0.0, source_y),
        v_fan      = v_fan_2d,
        t_max      = 2.0,
        hbar       = 0.07,
        n_steps    = 200,
        N_grid     = 150,
        integrator = 'verlet',
        equation   = EquationType.SCHRODINGER,
        parallel   = True,
        xlim       = (0.0, 4.0),
        ylim       = (-2.5, 2.5),
    )

print("Computing top slit...")
res_top = _slit_result(+d / 2)
print("Computing bottom slit...")
res_bottom = _slit_result(-d / 2)

# ─────────────────────────────────────────────────────────────────────────────
# PROPERLY COMBINE BOTH RESULTS
# ─────────────────────────────────────────────────────────────────────────────

import copy
result_ex4 = copy.copy(res_top)

# Wavefunction: COHERENT SUM (complex addition on the grid)
result_ex4.psi = res_top.psi + res_bottom.psi

# Rays: CONCATENATE both ray lists
result_ex4.rays = res_top.rays + res_bottom.rays

# Scattered data: CONCATENATE all point arrays (CRITICAL FIX)
result_ex4.x_pts = np.concatenate([res_top.x_pts, res_bottom.x_pts])
result_ex4.y_pts = np.concatenate([res_top.y_pts, res_bottom.y_pts])
result_ex4.S_pts = np.concatenate([res_top.S_pts, res_bottom.S_pts])
result_ex4.det_J_pts = np.concatenate([res_top.det_J_pts, res_bottom.det_J_pts])
result_ex4.mu_pts = np.concatenate([res_top.mu_pts, res_bottom.mu_pts])

# Grid coordinates (should be identical, but ensure consistency)
result_ex4.X = res_top.X
result_ex4.Y = res_top.Y

# Metadata
result_ex4.hbar = res_top.hbar
result_ex4.t_max = res_top.t_max
result_ex4.dim = res_top.dim
result_ex4.equation = res_top.equation

print(f"Total rays: {len(result_ex4.rays)} (top: {len(res_top.rays)}, bottom: {len(res_bottom.rays)})")
print(f"Total scattered points: {len(result_ex4.x_pts)}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOTTING
# ─────────────────────────────────────────────────────────────────────────────

fig4 = plot_wavefunction(result_ex4, log_scale=False)
fig4.suptitle(r"Ex 4 — 2D Schrödinger, double-slit  ($d=0.4$, $\hbar=0.07$)",
              color='white', fontsize=10, y=1.02)
plt.show()

fig_detail = plot_interference_detail(result_ex4)
fig_detail.suptitle(r"Ex 4 — Interference Detail", color='white', fontsize=10, y=1.02)
plt.show()

# Animation
anim = animate_wavefunction(result_ex4, n_frames=80, interval=40, log_scale=False)
HTML(anim.to_jshtml())  # Uncomment in Jupyter

## 2D - polar coordinates H = (p_r^2 + p_theta^2 / r^2) / 2

In [ ]:
# ----------------------------------------------------------------------
# Example 4: 2D polar coordinates H = (p_r^2 + p_theta^2 / r^2) / 2
#
# Three cautions specific to polar coordinates:
#
# (a) Coordinate singularity at r = 0: the metric component g^{θθ} = 1/r²
#     diverges.  Keep the source away from r=0 (r₀=1 is fine) and use
#     velocities small enough that no ray reaches r=0 during integration.
#
# (b) Coordinate singularity at θ = 0 / 2π for SymPy with positive=True:
#     evaluating g at θ=0 can trigger domain errors in lambdified
#     expressions.  A small offset θ₀ = 0.1 rad avoids this.
#
# (c) Zero-velocity rays: a fan that contains v=(0,0) produces a
#     degenerate constant trajectory; the Jacobi solver then receives a
#     trivially zero det J and the Maslov index computation is unreliable.
#     Build the fan from two 1D grids that exclude zero.
# ----------------------------------------------------------------------
r, theta = sp.symbols('r theta', real=True, positive=True)
pr, ptheta = sp.symbols('p_r p_theta', real=True)
H_2d = (pr**2 + ptheta**2 / r**2) / 2
metric_from_H_2d = Metric.from_hamiltonian(H_2d, (r, theta), (pr, ptheta))
hbar = 0.1
# (b) small θ offset so SymPy lambdify never evaluates at θ = 0
source_2d_h = (1.0, 0.1)

# (c) exclude zero: use linspace on strictly positive/negative halves
vr_pos  = np.linspace(0.05, 0.4, 40)
vr_vals = np.concatenate([-vr_pos[::-1], vr_pos])   # [-0.4,…,-0.05, 0.05,…,0.4]
vt_pos  = np.linspace(0.05, 0.4, 40)
vt_vals = np.concatenate([-vt_pos[::-1], vt_pos])
v_fan_2d_h = np.array([[a, b] for a in vr_vals for b in vt_vals])

t_max_2d_h = 1.5          # shorter: keeps rays well away from r = 0
result_2d_h = compute_wavefunction(
    metric=metric_from_H_2d,
    source=source_2d_h,
    v_fan=v_fan_2d_h,
    t_max=t_max_2d_h,
    hbar=hbar,
    n_steps=300,          # more steps for accuracy on curved metric
    N_grid=150,
    integrator='rk45',    # RK45 handles the r² denominator more robustly
    parallel   = True,
)
plot_wavefunction(result_2d_h, log_scale=True)
plt.show()

print("\nPlotting additional diagnostics for the 1D variable-mass example...")
plot_ray_fan(result_2d_h)
plt.show()
plot_interference_detail(result_2d_h)
plt.show()


anim = animate_wavefunction(result_2d_h, n_frames=80, interval=40)
HTML(anim.to_jshtml())

## 2D · Parabolic · curved metric  g = diag(1, 1 + x²)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ex 5 — 2D · Parabolic · curved metric  g = diag(1, 1 + x²)
# ─────────────────────────────────────────────────────────────────────────────
#
# PDE :  ∂u/∂t = ψOp u,   H = ½(ξ² + η²/(1+x²))
#
# The metric g = diag(1, 1+x²) is the induced metric on a paraboloid of
# revolution embedded in ℝ³.  The y-direction is "stretched" proportionally
# to x², slowing y-momentum transport away from x = 0.  This creates a
# non-trivial focusing geometry: rays are bent toward the y-axis by the
# effective refractive index √(1+x²).
#
# For the parabolic equation the solution is a real-valued density that
# diffuses anisotropically along the geodesics of the curved surface.
#
# What to observe:
#   • Anisotropic spreading: faster along x, slower along y away from origin.
#   • Curved ray paths visible in the ray-fan panel.
#   • log|u| shows the action "hillscape" over the paraboloid.
 
x, y = sp.symbols('x y', real=True)
g_paraboloid = sp.Matrix([[1, 0],
                           [0, 1 + x**2]])
metric_paraboloid = Metric(g_paraboloid, (x, y))
 
vx_vals = np.linspace(-2.0, 2.0, 20)
vy_vals = np.linspace(-2.0, 2.0, 20)
v_fan_para = np.array([[vx, vy] for vx in vx_vals for vy in vy_vals])
 
result_ex5 = compute_wavefunction(
    metric     = metric_paraboloid,
    source     = (0.0, 0.0),
    v_fan      = v_fan_para,
    t_max      = 1.8,
    hbar       = 0.1,
    n_steps    = 200,
    N_grid     = 150,
    integrator = 'verlet',
    equation   = EquationType.PARABOLIC,
    parallel   = True,
)
 
fig5 = plot_wavefunction(result_ex5, log_scale=True)
fig5.suptitle(r"Ex 5 — 2D Parabolic, curved metric  $g=\mathrm{diag}(1,\,1+x^2)$",
              color='white', fontsize=10)
plt.show()
 
fig5b = plot_ray_fan(result_ex5)
plt.show()

fig5c = plot_interference_detail(result_ex5)
plt.show()

anim = animate_wavefunction(result_ex5, n_frames=80, interval=40)
HTML(anim.to_jshtml())

## 2D · Wave equation · anisotropic flat metric  g = diag(1, 4)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ex 6 — 2D · Wave equation · anisotropic flat metric  g = diag(1, 4)
# ─────────────────────────────────────────────────────────────────────────────
#
# PDE :  ∂²u/∂t² = ψOp u,   H = ½(ξ² + η²/4)
#
# An anisotropic medium where the wave speed in the y-direction is halved
# (c_y = ½ c_x).  The dispersion branches H± = ±√(½(ξ² + η²/4)) generate
# elliptic wavefronts rather than circular ones.  The resulting interference
# pattern is stretched along y by a factor of 2 relative to the isotropic case.
#
# What to observe:
#   • Elliptic rather than circular wavefronts in the density panel.
#   • Two branch ray fans (forward + backward) visible as two concentric
#     elliptic families in the ray-fan panel.
#   • The aspect ratio of the fringe pattern is ≈ 2:1 (y elongated).
 
x, y = sp.symbols('x y', real=True)
g_aniso = sp.Matrix([[1, 0], [0, 4]])
metric_aniso = Metric(g_aniso, (x, y))
 
vx_vals = np.linspace(-2.5, 2.5, 10)
vy_vals = np.linspace(-2.5, 2.5, 10)
v_fan_aniso = np.array([[vx, vy] for vx in vx_vals for vy in vy_vals])
 
result_ex6 = compute_wavefunction(
    metric     = metric_aniso,
    source     = (0.0, 0.0),
    v_fan      = v_fan_aniso,
    t_max      = 2.0,
    hbar       = 0.1,
    n_steps    = 250,
    N_grid     = 150,
    integrator = 'verlet',
    equation   = EquationType.WAVE,
    parallel   = True,
)
 
fig6 = plot_wavefunction(result_ex6, log_scale=False)
fig6.suptitle(r"Ex 6 — 2D Wave equation, anisotropic metric  $g=\mathrm{diag}(1,4)$",
              color='white', fontsize=10)
plt.show()

fig6c = plot_interference_detail(result_ex6)
plt.show()

anim = animate_wavefunction(result_ex6, n_frames=80, interval=40)
HTML(anim.to_jshtml())

## 2D · Schrödinger · angular-momentum Hamiltonian  H = x·η − y·ξ

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ex 7 — 2D · Schrödinger · angular-momentum Hamiltonian  H = x·η − y·ξ
# ─────────────────────────────────────────────────────────────────────────────
#
# PDE :  −i ∂u/∂t = ψOp u,   H(x, y, ξ, η) = x·η − y·ξ
#
# This is the generator of rotations in the plane: the corresponding
# classical flow is rigid rotation at unit angular velocity,
#
#     ẋ =  ∂H/∂ξ = −y,     ξ̇ = −∂H/∂x = −η
#     ẏ =  ∂H/∂η =  x,     η̇ = −∂H/∂y =  ξ
#
# so every ray is a circle of radius r = √(x₀² + y₀²) traversed with period
# T = 2π.  The quantum operator is the z-component of angular momentum L_z.
#
# Because all rays are periodic circles the Jacobi determinant oscillates
# between 0 and its maximum at t = π, giving Maslov index μ = 2 after one
# full revolution.  The wavefunction accumulates a phase exp(−iπ) = −1 per
# half-turn — the Berry phase of the angular momentum eigenstates.
#
# What to observe:
#   • Perfectly circular ray fan (all rays are circles).
#   • Maslov scatter shows μ = 1 after half a revolution, μ = 2 after full.
#   • |ψ|² shows a ring structure (constructive interference on the circle).
#   • The phase map arg(ψ) winds by 2π around the origin — winding number = 1.
 
x, y   = sp.symbols('x y',   real=True)
xi, eta = sp.symbols('xi eta', real=True)
 
H_angular = (xi ** 2 + eta **2)/2 +  x * eta - y * xi       # L_z — generator of plane rotations
 
# Initialise rays on a ring of radius r₀ = 1.0, pointing tangentially
r0     = 1.0
n_rays = 30
theta  = np.linspace(0, 2 * np.pi, n_rays, endpoint=False)
 
# Positions on the ring
sources_ring = np.column_stack([r0 * np.cos(theta),
                                 r0 * np.sin(theta)])
 
# Tangential momenta (for L_z the tangential direction has p = (−sinθ, cosθ))
p_ring = np.column_stack([-np.sin(theta), np.cos(theta)])
 
# Compute one result per source and superpose (all sources on the ring)
# For efficiency we use a single representative source + a full momentum fan
# spanning all directions; this generates the same ring structure.
p_fan_ring = np.column_stack([
    np.linspace(-1.5, 1.5, 10),
    np.linspace(-1.5, 1.5, 10),
])
p_fan_full = np.array([[px, py]
                        for px in np.linspace(-1.5, 1.5, 25)
                        for py in np.linspace(-1.5, 1.5, 25)])
 
result_ex7 = compute_wavefunction(
    hamiltonian = H_angular,
    coords      = (x, y),
    momenta     = (xi, eta),
    source      = (r0, 0.0),          # single source on the ring
    p_fan       = p_fan_full,
    t_max       = 2 * np.pi,          # exactly one full revolution
    hbar        = 0.08,
    n_steps     = 300,
    N_grid      = 250,
    integrator  = 'rk45',
    equation    = EquationType.SCHRODINGER,
    parallel    = True,
    xlim        = (-2.5, 2.5),
    ylim        = (-2.5, 2.5),
)
 
fig7 = plot_wavefunction(result_ex7, log_scale=True)
fig7.suptitle(r"Ex 7 — 2D Schrödinger, $H = x\eta - y\xi$  (angular momentum / rotation)",
              color='white', fontsize=10)
plt.show()
 
fig7b = plot_ray_fan(result_ex7)
plt.show()
 
fig7c = plot_interference_detail(result_ex7)
fig7c.suptitle(r"Ex 7 — interference detail: $H = x\eta - y\xi$",
               color='white', fontsize=10)
plt.show()

anim = animate_wavefunction(result_ex7, n_frames=80, interval=40)
HTML(anim.to_jshtml())

## 2D · Schrödinger · unit sphere  ds² = dθ² + sin²θ dφ²

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ex 8 — 2D · Schrödinger · unit sphere  ds² = dθ² + sin²θ dφ²
# ─────────────────────────────────────────────────────────────────────────────
#
# PDE :  −i ∂ψ/∂t = −½ Δ_S² ψ
#
# where Δ_S² is the Laplace–Beltrami operator on the unit 2-sphere with
# metric g = diag(1, sin²θ).  The canonical momenta are (p_θ, p_φ) and the
# Hamiltonian is the geodesic kinetic energy:
#
#     H = ½ (p_θ² + p_φ²/sin²θ)
#
# Geodesics are great circles.  All great circles leaving a point reconverge
# at the antipodal point after an arc length of π, creating a perfect
# global caustic — a spherical analogue of a rainbow.
#
# Coordinate conventions
# ~~~~~~~~~~~~~~~~~~~~~~
# θ  ∈ (0, π)   — colatitude (0 = north pole, π = south pole)
# φ  ∈ (−π, π)  — longitude
#
# The north pole θ = 0 is a coordinate singularity (sin θ = 0, g_φφ = 0),
# so the source is placed slightly away from it at θ₀ = 0.15 rad ≈ 8.6°.
# t_max is set to 1.8 < π so rays have not yet reached the antipodal caustic;
# the wavefunction shows the beginning of the focusing process.
#
# What to observe
# ~~~~~~~~~~~~~~~
# • All geodesics (great circles) are curved lines in (θ, φ) space — the
#   curvature of the sphere bends them together.
# • Rays fan out from the source and begin converging toward θ ≈ π (south
#   pole) but have not yet arrived: det J decreases toward the caustic.
# • The phase map arg(ψ) shows the eikonal S = ∫ p·dq accumulating along
#   the great-circle arcs.
# • Maslov index μ = 0 everywhere (no caustic crossing yet for t_max < π).
# • The density |ψ|² is a ring-like pattern expanding away from the source,
#   modulated by the 1/sin θ focusing factor of the spherical geometry.

# ── metric ────────────────────────────────────────────────────────────────────
theta, phi = sp.symbols('theta phi', real=True)

g_sphere = Matrix([[1,               0            ],
                   [0, sin(theta)**2              ]])

metric_sphere = Metric(g_sphere, (theta, phi))

# ── ray fan ───────────────────────────────────────────────────────────────────
# Source slightly off the north pole to avoid the coordinate singularity.
# Initial velocities span all azimuthal directions uniformly:
#   v_θ = cos(α),  v_φ = sin(α)  for α ∈ [0, 2π)
# These are velocities in the local orthonormal frame; converting to
# coordinate velocities:
#   ṫheta = v_θ / √g_θθ = v_θ / 1 = v_θ
#   ṗhi   = v_φ / √g_φφ = v_φ / sin(θ₀)
# The `v_fan` parameter of compute_wavefunction expects coordinate velocities
# [dθ/dt, dφ/dt], so we divide v_φ by sin(θ₀) to get unit-speed geodesics.

theta0 = 0.15          # source colatitude (≈ 8.6° from north pole)
phi0   = 0.0           # source longitude
source = (theta0, phi0)

n_rays  = 100           # one ray every 5°
azimuths = np.linspace(0, 2 * np.pi, n_rays, endpoint=False)
sin_th0  = float(sp.sin(theta0).evalf())

v_fan_sphere = np.column_stack([
    np.cos(azimuths),              # dθ/dt  — unit colatitude velocity
    np.sin(azimuths) / sin_th0,    # dφ/dt  — scaled so |v|_g = 1
])

# ── compute ───────────────────────────────────────────────────────────────────
result_ex8 = compute_wavefunction(
    metric     = metric_sphere,
    source     = source,
    v_fan      = v_fan_sphere,
    t_max      = 1.8,          # < π — rays still converging, no south-pole caustic
    hbar       = 0.08,
    n_steps    = 250,
    N_grid     = 150,
    integrator = 'rk45',       # Verlet can struggle with the sin²θ curvature
    equation   = EquationType.SCHRODINGER,
    parallel   = True,
    xlim       = (0.0,  np.pi),         # θ range
    ylim       = (-np.pi, np.pi),       # φ range
)

# ── plot ──────────────────────────────────────────────────────────────────────
fig8 = plot_wavefunction(result_ex8, log_scale=True)
fig8.suptitle(
    r"Ex 8 — 2D Schrödinger on the unit sphere  $ds^2 = d\theta^2 + \sin^2\!\theta\, d\phi^2$"
    "\n"
    r"Source at $(\theta_0, \phi_0) = (0.15,\, 0)$,  $t_{\max} = 1.8 < \pi$",
    color='white', fontsize=9,
)
plt.show()

fig8b = plot_ray_fan(result_ex8)
fig8b.axes[0].set(
    xlabel=r"$\phi$  (longitude)",
    ylabel=r"$\theta$  (colatitude)",
    title="Great-circle ray fan on the sphere  (yellow = caustic onset)",
)
plt.show()

fig8c = plot_interference_detail(result_ex8)
fig8c.suptitle(
    r"Ex 8 — interference detail: sphere  $\theta$–$\phi$ coordinates",
    color='white', fontsize=9,
)
plt.show()


anim = animate_wavefunction(result_ex8, n_frames=80, interval=40)
HTML(anim.to_jshtml())

## 2D Harmonic Oscillator — Non-constant Maslov Index

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Example: Harmonic Oscillator — Non-constant Maslov Index
# ─────────────────────────────────────────────────────────────────────────────
#
# H = ½(p_x² + p_y²) + ½(x² + y²)
#
# Key physics:
#   • Rays focus at t = π, 2π, 3π, ... (periodic caustics)
#   • Each focus increments Maslov index μ by +1
#   • With t_max > π, different rays have different μ values
#   • This creates spatial variation in the Maslov index plot
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from propagator import compute_wavefunction, plot_wavefunction, plot_ray_fan
from propagator import EquationType

# Define 2D harmonic oscillator Hamiltonian
x, y, px, py = sp.symbols('x y px py', real=True)
H_ho = (px**2 + py**2) / 2 + (x**2 + y**2) / 2

# Source at origin
source = (0.0, 0.0)

# Momentum fan - exclude zero to avoid degenerate rays
p_mag = np.linspace(0.5, 2.0, 30)
p_ang = np.linspace(0, 2*np.pi, 40, endpoint=False)
p_fan = np.array([[p * np.cos(theta), p * np.sin(theta)] 
                  for p in p_mag for theta in p_ang])

print(f"Total rays: {len(p_fan)}")

# ─────────────────────────────────────────────────────────────────────────────
# Case 1: t_max < π  →  No caustics, μ = 0 everywhere
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("Case 1: t_max = 2.0  (< π)  →  No caustics, μ = 0 everywhere")
print("="*70)

result_1 = compute_wavefunction(
    hamiltonian = H_ho,
    coords      = (x, y),
    momenta     = (px, py),
    source      = source,
    p_fan       = p_fan,
    t_max       = 2.0,          # Less than π ≈ 3.14
    hbar        = 0.1,
    n_steps     = 200,
    N_grid      = 100,
    integrator  = 'rk45',
    equation    = EquationType.SCHRODINGER,
    parallel    = True,
)

print(f"Maslov index range: [{result_1.mu_pts.min()}, {result_1.mu_pts.max()}]")
print(f"Unique Maslov values: {np.unique(result_1.mu_pts)}")

fig1 = plot_wavefunction(result_1, log_scale=True)
fig1.suptitle(r"HO 2D: $t_{max}=2.0 < \pi$  (No caustics, $\mu=0$)", 
              color='white', fontsize=10, y=1.02)
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Case 2: t_max > π  →  Caustics formed, μ ≥ 1 for some rays
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("Case 2: t_max = 4.0  (> π)  →  Caustics formed, μ varies")
print("="*70)

result_2 = compute_wavefunction(
    hamiltonian = H_ho,
    coords      = (x, y),
    momenta     = (px, py),
    source      = source,
    p_fan       = p_fan,
    t_max       = 4.0,          # Greater than π ≈ 3.14
    hbar        = 0.1,
    n_steps     = 250,          # More steps for accuracy through caustic
    N_grid      = 150,
    integrator  = 'rk45',
    equation    = EquationType.SCHRODINGER,
    parallel    = True,
)

print(f"Maslov index range: [{result_2.mu_pts.min()}, {result_2.mu_pts.max()}]")
print(f"Unique Maslov values: {np.unique(result_2.mu_pts)}")
print(f"Rays with μ=0: {np.sum(result_2.mu_pts == 0)}")
print(f"Rays with μ=1: {np.sum(result_2.mu_pts == 1)}")
print(f"Rays with μ≥2: {np.sum(result_2.mu_pts >= 2)}")

fig2 = plot_wavefunction(result_2, log_scale=True)
fig2.suptitle(r"HO 2D: $t_{max}=4.0 > \pi$  (Caustics, $\mu \geq 1$)", 
              color='white', fontsize=10, y=1.02)
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Case 3: t_max > 2π  →  Multiple caustic crossings, μ ≥ 2
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("Case 3: t_max = 7.0  (> 2π)  →  Multiple caustics, μ ≥ 2")
print("="*70)

result_3 = compute_wavefunction(
    hamiltonian = H_ho,
    coords      = (x, y),
    momenta     = (px, py),
    source      = source,
    p_fan       = p_fan,
    t_max       = 7.0,          # Greater than 2π ≈ 6.28
    hbar        = 0.1,
    n_steps     = 250,
    N_grid      = 1500,
    integrator  = 'rk45',
    equation    = EquationType.SCHRODINGER,
    parallel    = True,
)

print(f"Maslov index range: [{result_3.mu_pts.min()}, {result_3.mu_pts.max()}]")
print(f"Unique Maslov values: {np.unique(result_3.mu_pts)}")

fig3 = plot_wavefunction(result_3, log_scale=True)
fig3.suptitle(r"HO 2D: $t_{max}=7.0 > 2\pi$  (Multiple caustics, $\mu \geq 2$)", 
              color='white', fontsize=10, y=1.02)
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Diagnostic: Plot Maslov index distribution
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, result, t_max in zip(axes, [result_1, result_2, result_3], [2.0, 4.0, 7.0]):
    unique, counts = np.unique(result.mu_pts, return_counts=True)
    ax.bar(unique, counts, color='steelblue', alpha=0.7)
    ax.set_xlabel(r"Maslov index $\mu$ ")
    ax.set_ylabel("Number of ray points")
    ax.set_title(f"t_max = {t_max}")
    ax.set_xticks(unique)

plt.tight_layout()
fig.patch.set_facecolor('#0e0e1a')
for ax in axes:
    ax.set_facecolor('#0e0e1a')
    ax.tick_params(colors='white')
    for lbl in (ax.xaxis.label, ax.yaxis.label, ax.title):
        lbl.set_color('white')
    for sp in ax.spines.values():
        sp.set_edgecolor('#444')

plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Diagnostic: Show rays coloured by Maslov index
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))

x_key, y_key = 'x', 'y'
cmap = plt.cm.RdYlGn

for ray in result_2.rays:
    ax.plot(ray.traj[x_key], ray.traj[y_key], 
            lw=0.5, alpha=0.3, 
            color=cmap(ray.mu / max(1, result_2.mu_pts.max())))

ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(r"HO 2D: Rays coloured by Maslov index $\mu$  ($t_{max}=4.0$)")
ax.set_facecolor('#0e0e1a')
ax.tick_params(colors='white')
for lbl in (ax.xaxis.label, ax.yaxis.label, ax.title):
    lbl.set_color('white')
for sp in ax.spines.values():
    sp.set_edgecolor('#444')

plt.show()